# Laboratorio Semana 6: Preparación de Datos y Benchmarking (CSV vs Parquet)
**Materia:** Fundamentos de Big Data (IDSD413) - Escuela Politécnica Nacional
**Profesor:** David Esteban Galarza García

En este laboratorio demostraremos empíricamente el problema del cuello de botella de I/O (Lectura/Escritura) en Big Data. Generaremos un dataset sintético masivo de transacciones de retail y compararemos el rendimiento de un formato basado en filas (CSV) frente a un formato columnar nativo de Big Data (Apache Parquet).

In [13]:
import pandas as pd
import numpy as np
import time
import os

# Configuración del volumen de datos (15 millones de registros)
N_ROWS = 15_000_000

print(f"Generando {N_ROWS} registros en memoria RAM...")
start_time = time.time()

# Creación de un dataset sintético de transacciones de retail
np.random.seed(42)
df = pd.DataFrame({
    'id_transaccion': np.arange(N_ROWS),
    'id_cliente': np.random.randint(1, 100000, N_ROWS),
    'monto': np.random.uniform(5.0, 2500.0, N_ROWS),
    'categoria': np.random.choice(['Electronica', 'Ropa', 'Hogar', 'Alimentos', 'Juguetes'], N_ROWS),
    'estado': np.random.choice(['Completado', 'Pendiente', 'Cancelado', 'Reembolsado'], N_ROWS)
})

print(f"Dataset generado en {time.time() - start_time:.2f} segundos.")
df.info()

Generando 15000000 registros en memoria RAM...
Dataset generado en 10.57 segundos.
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000000 entries, 0 to 14999999
Data columns (total 5 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id_transaccion  int64  
 1   id_cliente      int64  
 2   monto           float64
 3   categoria       object 
 4   estado          object 
dtypes: float64(1), int64(2), object(2)
memory usage: 572.2+ MB


## 1. Prueba de Escritura y Compresión en Disco
Guardaremos exactamente el mismo DataFrame en ambos formatos para analizar los tiempos de escritura y la eficiencia del algoritmo de compresión (Snappy) que utiliza Parquet por defecto.

In [14]:
# 1.1 Escritura en CSV (Basado en filas, texto plano)
print("Escribiendo archivo CSV...")
start_csv_write = time.time()
df.to_csv('transacciones_epn.csv', index=False)
csv_write_time = time.time() - start_csv_write
print(f"Tiempo de escritura CSV: {csv_write_time:.2f} segundos")

# 1.2 Escritura en Parquet (Columnar, tipado fuerte, comprimido)
print("\nEscribiendo archivo Parquet...")
start_parquet_write = time.time()
df.to_parquet('transacciones_epn.parquet', engine='pyarrow')
parquet_write_time = time.time() - start_parquet_write
print(f"Tiempo de escritura Parquet: {parquet_write_time:.2f} segundos")

# 1.3 Comparación de peso en el almacenamiento
csv_size = os.path.getsize('transacciones_epn.csv') / (1024 * 1024)
parquet_size = os.path.getsize('transacciones_epn.parquet') / (1024 * 1024)

print("-" * 40)
print(f"Peso del CSV: {csv_size:.2f} MB")
print(f"Peso del Parquet: {parquet_size:.2f} MB")
print(f"Reducción de almacenamiento: {((csv_size - parquet_size) / csv_size) * 100:.2f}%")

Escribiendo archivo CSV...
Tiempo de escritura CSV: 76.59 segundos

Escribiendo archivo Parquet...
Tiempo de escritura Parquet: 7.37 segundos
----------------------------------------
Peso del CSV: 740.53 MB
Peso del Parquet: 225.39 MB
Reducción de almacenamiento: 69.56%


## 2. Prueba de Lectura Completa (Full Scan)
Evaluaremos cuánto tarda el procesador en levantar la totalidad del archivo desde el disco duro hacia la memoria RAM de Google Colab.

In [15]:
# Limpiamos la memoria para evitar sesgos en la prueba
del df
import gc
gc.collect()

# 2.1 Lectura de CSV
print("Leyendo CSV a memoria...")
start_csv_read = time.time()
df_csv = pd.read_csv('transacciones_epn.csv')
csv_read_time = time.time() - start_csv_read
print(f"Tiempo de lectura CSV: {csv_read_time:.2f} segundos")

# 2.2 Lectura de Parquet
print("\nLeyendo Parquet a memoria...")
start_parquet_read = time.time()
df_parquet = pd.read_parquet('transacciones_epn.parquet', engine='pyarrow')
parquet_read_time = time.time() - start_parquet_read
print(f"Tiempo de lectura Parquet: {parquet_read_time:.2f} segundos")

Leyendo CSV a memoria...
Tiempo de lectura CSV: 12.51 segundos

Leyendo Parquet a memoria...
Tiempo de lectura Parquet: 3.11 segundos


In [16]:
import psutil, os, gc
proc = psutil.Process(os.getpid())
ram = lambda: proc.memory_info().rss / 2**20

# Limpiar lo que quedó cargado
for v in ['df_csv', 'df_parquet']:
    if v in globals(): del globals()[v]
gc.collect()
print(f"RAM base: {ram():.0f} MB")

# CSV completo
df_csv = pd.read_csv('transacciones_epn.csv')
print(f"Después de leer CSV completo: {ram():.0f} MB")
print(f"Tamaño del DataFrame CSV: {df_csv.memory_usage(deep=True).sum()/2**20:.0f} MB")
del df_csv; gc.collect()
print(f"Tras liberar: {ram():.0f} MB")

# Parquet solo columna monto
df_pq = pd.read_parquet('transacciones_epn.parquet', columns=['monto'])
print(f"Después de leer Parquet (1 columna): {ram():.0f} MB")
print(f"Tamaño del DataFrame Parquet: {df_pq.memory_usage(deep=True).sum()/2**20:.0f} MB")
del df_pq; gc.collect()

RAM base: 1271 MB
Después de leer CSV completo: 1917 MB
Tamaño del DataFrame CSV: 1991 MB
Tras liberar: 1345 MB
Después de leer Parquet (1 columna): 1402 MB
Tamaño del DataFrame Parquet: 114 MB


0

## 3. Ventaja Columnar: Consultas Selectivas (Column Pruning)
En Ciencia de Datos, rara vez analizamos todas las columnas a la vez. En este escenario, el equipo de analítica solo necesita calcular el total de ventas (columna `monto`).

Veremos cómo Parquet escanea *únicamente* los bytes de esa columna, mientras que CSV está obligado a leer el archivo entero.

In [17]:
# 3.1 Agregación selectiva en CSV
print("Calculando ventas totales desde CSV...")
start_csv_query = time.time()
# Pandas tiene que analizar todo el archivo para extraer una sola columna
suma_csv = pd.read_csv('transacciones_epn.csv', usecols=['monto'])['monto'].sum()
csv_query_time = time.time() - start_csv_query
print(f"Ventas (CSV): ${suma_csv:,.2f} | Tiempo: {csv_query_time:.4f} segundos")

# 3.2 Agregación selectiva en Parquet
print("\nCalculando ventas totales desde Parquet...")
start_parquet_query = time.time()
# PyArrow inyecta la proyección directamente a nivel de disco
suma_parquet = pd.read_parquet('transacciones_epn.parquet', columns=['monto'], engine='pyarrow')['monto'].sum()
parquet_query_time = time.time() - start_parquet_query
print(f"Ventas (Parquet): ${suma_parquet:,.2f} | Tiempo: {parquet_query_time:.4f} segundos")

print("-" * 40)
print(f"Aceleración de consulta selectiva: {csv_query_time / parquet_query_time:.2f} veces más rápido")

Calculando ventas totales desde CSV...
Ventas (CSV): $18,792,792,944.85 | Tiempo: 13.0181 segundos

Calculando ventas totales desde Parquet...
Ventas (Parquet): $18,792,792,944.85 | Tiempo: 0.4028 segundos
----------------------------------------
Aceleración de consulta selectiva: 32.32 veces más rápido


## 4. Reto en Vivo 1: Filtrado desde el Disco (Predicate Pushdown)
¿Qué pasa si nuestro modelo solo necesita los datos de 'Electronica'?
En CSV, la memoria RAM puede colapsar porque Pandas debe cargar todo el archivo antes de filtrar. En Parquet, el motor filtra los datos directamente en el disco duro y solo envía a la RAM lo que coincide.

In [18]:
# 1. El método tradicional (CSV)
print("Filtrando CSV (Carga total -> Filtro)...")
start_csv = time.time()
df_temp_csv = pd.read_csv('transacciones_epn.csv')
df_filtrado_csv = df_temp_csv[df_temp_csv['categoria'] == 'Juguetes']
tiempo_csv = time.time() - start_csv
print(f"Tiempo CSV: {tiempo_csv:.4f} segundos | Filas: {len(df_filtrado_csv)}")

# Liberamos memoria
del df_temp_csv, df_filtrado_csv
import gc; gc.collect()

# 2. El método Big Data (Parquet con Predicate Pushdown)
print("\nFiltrando Parquet (Filtro en disco -> Carga parcial)...")
start_parquet = time.time()
df_filtrado_parquet = pd.read_parquet(
    'transacciones_epn.parquet',
    engine='pyarrow',
    filters=[('categoria', '==', 'Juguetes')]
)
tiempo_parquet = time.time() - start_parquet
print(f"Tiempo Parquet: {tiempo_parquet:.4f} segundos | Filas: {len(df_filtrado_parquet)}")

print("-" * 40)
print(f"Aceleración: {tiempo_csv / tiempo_parquet:.2f}x más rápido")

Filtrando CSV (Carga total -> Filtro)...
Tiempo CSV: 17.3385 segundos | Filas: 2999407

Filtrando Parquet (Filtro en disco -> Carga parcial)...
Tiempo Parquet: 2.3061 segundos | Filas: 2999407
----------------------------------------
Aceleración: 7.52x más rápido


In [19]:
import gc

# CSV: carga todo y luego filtra
print("Filtrando CSV (carga total -> filtro)...")
t0 = time.time()
d = pd.read_csv('transacciones_epn.csv')
r_csv = d.loc[(d.categoria == 'Juguetes') & (d.estado == 'Completado'), 'monto']
t_csv2 = time.time() - t0
print(f"CSV: {t_csv2:.2f}s | filas={len(r_csv):,} | suma=${r_csv.sum():,.2f}")
del d; gc.collect()

# Parquet: filtros en disco + solo la columna monto
print("\nFiltrando Parquet (Predicate Pushdown)...")
t0 = time.time()
r_pq = pd.read_parquet(
    'transacciones_epn.parquet', engine='pyarrow',
    columns=['monto'],
    filters=[('categoria', '==', 'Juguetes'), ('estado', '==', 'Completado')]
)
t_pq2 = time.time() - t0
print(f"Parquet: {t_pq2:.2f}s | filas={len(r_pq):,} | suma=${r_pq['monto'].sum():,.2f}")

print("-" * 40)
print(f"Aceleración: {t_csv2 / t_pq2:.1f}x más rápido")

Filtrando CSV (carga total -> filtro)...
CSV: 15.88s | filas=750,894 | suma=$939,863,458.14

Filtrando Parquet (Predicate Pushdown)...
Parquet: 1.67s | filas=750,894 | suma=$939,863,458.14
----------------------------------------
Aceleración: 9.5x más rápido


## 5. Reto en Vivo 2: Agrupación y Estadísticas (Group By)
El análisis exploratorio requiere agrupar datos. Veamos cómo impacta la lectura columnar al calcular el promedio de ventas agrupado por el estado de la transacción.

In [20]:
# 1. Agrupación en CSV
print("Agrupando desde CSV...")
start_csv_grp = time.time()
grupo_csv = pd.read_csv('transacciones_epn.csv').groupby('estado')['monto'].mean()
tiempo_csv_grp = time.time() - start_csv_grp
print(f"Tiempo CSV: {tiempo_csv_grp:.4f} segundos")

# 2. Agrupación en Parquet
print("\nAgrupando desde Parquet...")
start_parquet_grp = time.time()
grupo_parquet = pd.read_parquet(
    'transacciones_epn.parquet',
    columns=['estado', 'monto'],
    engine='pyarrow'
).groupby('estado')['monto'].mean()
tiempo_parquet_grp = time.time() - start_parquet_grp
print(f"Tiempo Parquet: {tiempo_parquet_grp:.4f} segundos")

print("-" * 40)
print("Resultados de la agregación:")
print(grupo_parquet)

Agrupando desde CSV...
Tiempo CSV: 15.4408 segundos

Agrupando desde Parquet...
Tiempo Parquet: 4.1136 segundos
----------------------------------------
Resultados de la agregación:
estado
Cancelado      1252.851287
Completado     1252.511521
Pendiente      1253.266994
Reembolsado    1252.781580
Name: monto, dtype: float64


In [21]:
import pyarrow.parquet as pq
pf = pq.ParquetFile('transacciones_epn.parquet')
rg = pf.metadata.row_group(0)
print("Tamaño comprimido por columna:")
for i in range(rg.num_columns):
    c = rg.column(i)
    print(f"  {c.path_in_schema:15s} {c.total_compressed_size/2**20:8.2f} MB")
print("\nColumnas leídas en la consulta: monto + categoria + estado = 3 de 5")

Tamaño comprimido por columna:
  id_transaccion      4.26 MB
  id_cliente          2.55 MB
  monto               8.26 MB
  categoria           0.38 MB
  estado              0.26 MB

Columnas leídas en la consulta: monto + categoria + estado = 3 de 5
